# Day 2: Building and Training Our First Model

**Goal:** Construct and train our baseline CNN model.

### 1. Import Libraries

In [13]:
import numpy as np
import os
from tensorflow import keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

### 2. Load Data

Now, we'll load the preprocessed and split data that we saved in the previous notebook. These files contain our training, validation, and testing sets.

In [14]:
# Load the preprocessed data
X_train = np.load('../results/X_train.npy')
y_train = np.load('../results/y_train.npy')
X_val = np.load('../results/X_val.npy')
y_val = np.load('../results/y_val.npy')
X_test = np.load('../results/X_test.npy')
y_test = np.load('../results/y_test.npy')

### 3. Build the CNN Model

Now we will define the architecture of our CNN. A typical CNN for image classification consists of:

*   **Convolutional Layers (`Conv2D`):** These layers apply filters to the input image to extract features like edges, textures, and shapes. We'll use the 'relu' activation function to introduce non-linearity.
*   **Max Pooling Layers (`MaxPooling2D`):** These layers downsample the feature maps, reducing the spatial dimensions and making the model more efficient and robust to variations in the position of features.
*   **Flatten Layer:** This layer converts the 2D feature maps into a 1D vector, preparing the data for the fully connected layers.
*   **Dense Layers:** These are fully connected layers that perform classification based on the features extracted by the convolutional layers.
*   **Dropout Layer:** This layer helps prevent overfitting by randomly setting a fraction of input units to 0 at each update during training.
*   **Output Layer:** The final dense layer with a 'softmax' activation function, which outputs a probability distribution over the different tumor classes.

In [15]:
model = Sequential([
    keras.Input(shape=(150, 150, 3)),
    # First convolutional block
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Second convolutional block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Third convolutional block
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    
    # Dense layer for classification
    Dense(512, activation='relu'),
    Dropout(0.5),
    
    # Output layer with softmax activation for multi-class classification
    Dense(4, activation='softmax')  # 4 classes: glioma, meningioma, notumor, pituitary
])

model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 74, 74, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_7 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 max_pooling2d_7 (MaxPoolin  (None, 36, 36, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_8 (Conv2D)           (None, 34, 34, 128)       73856     
                                                                 
 max_pooling2d_8 (MaxPoolin  (None, 17, 17, 128)      

### 4. Compile the Model

Before we can train the model, we need to configure the learning process. This is done via the `compile` method, which takes three important arguments:

*   **Optimizer:** This is the algorithm that will be used to update the weights of the model to minimize the loss. A good and popular choice is `'adam'`.
*   **Loss Function:** This measures how accurate the model is during training. We want to minimize this function. Since we have multiple classes and our labels are one-hot encoded, we should use `'categorical_crossentropy'`.
*   **Metrics:** This is a list of metrics to be evaluated by the model during training and testing. For classification problems, a good choice is `['accuracy']` to see the percentage of images that are correctly classified.

In [16]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])  

### 5. Train the Model

Now for the exciting part! We will train our model using the `fit` method. This is where the model will learn from our training data.

*   We provide the training data (`X_train`, `y_train`).
*   `epochs=30`: An epoch is one complete pass through the entire training dataset. We'll train for 30 epochs.
*   `validation_data=(X_val, y_val)`: After each epoch, the model will evaluate its performance on the validation set. This is crucial for monitoring overfitting. If the training accuracy keeps going up but the validation accuracy flattens or goes down, it's a sign of overfitting.
*   The `fit` method returns a `history` object, which contains a record of the training and validation loss and accuracy at each epoch. We will store this so we can plot the learning curves later.

In [17]:
# The model expects 3-channel images (RGB), but our data is currently 1-channel (grayscale).
# We can fix this by repeating the grayscale channel three times.
X_train = np.repeat(X_train, 3, axis=-1)
X_val = np.repeat(X_val, 3, axis=-1)
X_test = np.repeat(X_test, 3, axis=-1)

# Let's check the new shape
print("Updated X_train shape:", X_train.shape)

Updated X_train shape: (4480, 150, 150, 3)


In [18]:
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_val, y_val))

Epoch 1/30
140/140 [==============================] - 18s 123ms/step - loss: 0.7736 - accuracy: 0.6862 - val_loss: 0.4444 - val_accuracy: 0.8348
Epoch 2/30
140/140 [==============================] - 18s 128ms/step - loss: 0.4289 - accuracy: 0.8344 - val_loss: 0.4221 - val_accuracy: 0.8161
Epoch 3/30
140/140 [==============================] - 18s 131ms/step - loss: 0.2952 - accuracy: 0.8842 - val_loss: 0.3081 - val_accuracy: 0.9009
Epoch 4/30
140/140 [==============================] - 17s 123ms/step - loss: 0.2292 - accuracy: 0.9141 - val_loss: 0.2808 - val_accuracy: 0.9000
Epoch 5/30
140/140 [==============================] - 17s 120ms/step - loss: 0.1516 - accuracy: 0.9451 - val_loss: 0.2202 - val_accuracy: 0.9411
Epoch 6/30
140/140 [==============================] - 17s 124ms/step - loss: 0.0973 - accuracy: 0.9634 - val_loss: 0.2616 - val_accuracy: 0.9268
Epoch 7/30
140/140 [==============================] - 18s 126ms/step - loss: 0.0810 - accuracy: 0.9690 - val_loss: 0.2474 - val_ac

### 6. Save the Trained Model

Finally, we'll save our trained model. This will save the model's architecture, weights, and training configuration to a single file. We can then load this file later to make predictions on new data without having to retrain the model.

In [20]:
# Save the model to the results directory
model.save('../results/baseline_model.keras')

print("Model saved successfully to '../results/baseline_model.keras'")

Model saved successfully to '../results/baseline_model.keras'
